# Tutorial 12-1: Tuning the Linear Perceptron on Non-Linear Data
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
We implement a single-layer **Perceptron** using PyTorch to classify the **Sonar dataset**. This dataset involves discriminating between sonar signals bounced off a metal cylinder (Mine) and a rock. We will tune the learning rate to find the maximum possible accuracy for a linear model.

### Theoretical Context
A Perceptron uses a linear combination of inputs to create a **linear decision boundary**. Because the Perceptron learning rule is a stochastic gradient descent method, it will converge on a hyperplane that minimizes error. However, if the data is non-linearly separable—like the XOR problem—the linear model will hit a performance ceiling.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Load and Preprocess Sonar Data
sonar = fetch_openml(name='sonar', version=1, as_frame=False)
X, y = sonar.data, sonar.target

# Encode targets (M/R) to 0/1
le = LabelEncoder()
y = le.fit_transform(y)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.FloatTensor(y_train).view(-1, 1)
y_test = torch.FloatTensor(y_test).view(-1, 1)

## 2. Tuning the Perceptron
We will iterate through multiple learning rates ($\lambda$) to find the best fit for our linear model.

In [ ]:
class Perceptron(nn.Module):
    def __init__(self, input_dim):
        super(Perceptron, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.linear(x))

lrs = [0.5, 0.1, 0.01, 0.001]
best_acc = 0

for lr in lrs:
    model = Perceptron(X_train.shape[1])
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    
    for epoch in range(200):
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    with torch.no_grad():
        acc = (model(X_test).round().eq(y_test).sum() / float(len(y_test))).item()
        print(f"LR: {lr} | Test Acc: {acc:.4f}")
        best_acc = max(best_acc, acc)

print(f"\nBest Perceptron Performance: {best_acc:.4f}")

## Conclusion
The Perceptron establishes a baseline but struggles with the complex patterns in the Sonar data. Because a single output node only sums inputs according to link weights, it cannot capture higher-order relationships between sensors. In 12-2, we add a hidden layer to break this linear constraint.